# 特徴量エンジニアリング — 可視化付き探索

目的: `docs/modeling/feature_engineering_exploration.md` の候補（例: **レース内圧力場**）を、
ローカル flat Parquet を読みながら **分布・年次ドリフト・仮説特徴** を可視化し、新規要素の当たりを付ける。

- **LayerA**: 出馬表・指数など T-10 で観測可能な列のみ（オッズ由来は別ノート）。
- データ無し環境でもセルが落ちないよう分岐あり。先に `data/local/tables/<年>/*_flat.parquet` を用意すること。

## 0. 環境セットアップ

**初回のみ**（または venv に未導入のとき）次のいずれかを実行。

```bash
# リポジトリルートで
python -m venv .venv && source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -U pip
pip install -r requirements.txt
pip install -r notebooks/feature_engineering/requirements.txt
python -m ipykernel install --user --name keiba-vpn --display-name "Python (keiba-vpn)"
```

Jupyter で本ノートを開き、カーネルに **Python (keiba-vpn)** を選ぶ。

In [ ]:
# ノート専用パッケージの不足をその場で補う（ルート requirements に含まれないもの）
%pip install -q seaborn jupyterlab ipykernel

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pipeline" / "feature_store.py").exists():
            return p.resolve()
    return start.resolve()

HERE = Path.cwd().resolve()
REPO_ROOT = find_repo_root(HERE)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
os.chdir(REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.pipeline.features.feature_store import FeatureStore

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Noto Sans JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 1. データ可用性とスキーマ

`FeatureStore` が参照する `data/local/tables/` の有無を確認し、`race_index` に数値指数列があるか調べる。

In [ ]:
store = FeatureStore(base_dir=REPO_ROOT)
years = store.available_years()
print("available_years:", years)

if not years:
    print("\n⚠ data/local/tables/<年>/ が空です。export_tables 等で Parquet を配置してから再実行。")
else:
    y0 = years[-1]
    print(f"\n例: {y0} のソース一覧（一部）", store.list_sources(y0)[:12], "...")
    if "race_index" in store.list_sources(y0):
        schema = store.source_schema("race_index", year=y0)
        num_cols = [c for c, t in schema.items() if "int" in t.lower() or "float" in t.lower() or "double" in t.lower()]
        print("\nrace_index 数値列（先頭30）:", num_cols[:30])

## 2. 読み込み（軽量: 必要列のみ）

指数の **レース内分散**（混戦度）の素案: `speed_max` が無ければ名前に `speed` を含む最初の数値列を使う。

In [ ]:
EXPLORE_YEARS = ["2022", "2023", "2024"]  # 必要に応じて変更（存在する年のみ）
EXPLORE_YEARS = [y for y in EXPLORE_YEARS if y in years]

df_idx = pd.DataFrame()
speed_col = None

if EXPLORE_YEARS and "race_index" in store.list_sources(EXPLORE_YEARS[0]):
    sch = store.source_schema("race_index", year=EXPLORE_YEARS[0])
    candidates = [c for c in sch if "speed" in c.lower()]
    if "speed_max" in sch:
        speed_col = "speed_max"
    elif candidates:
        speed_col = candidates[0]
    cols = ["race_id", "horse_number", speed_col] if speed_col else ["race_id", "horse_number"]
    df_idx = store.load_source("race_index", years=EXPLORE_YEARS, columns=[c for c in cols if c])
    if speed_col and speed_col in df_idx.columns:
        df_idx[speed_col] = pd.to_numeric(df_idx[speed_col], errors="coerce")
    print("rows:", len(df_idx), "speed_col:", speed_col)
    display(df_idx.head())
else:
    print("race_index をスキップ（データまたはカテゴリなし）")

## 3. 仮説特徴: `field_speed_spread`（レース内 std）

探索メモの `field_speed_spread` に相当。各馬行に **同一 race_id 内の指数 std** を付与（リークなし・出走メンバー依存）。

In [ ]:
if df_idx.empty or not speed_col or speed_col not in df_idx.columns:
    print("field 特徴は計算スキップ")
else:
    gstd = df_idx.groupby("race_id")[speed_col].transform("std")
    df_idx["field_speed_spread"] = gstd
    df_idx["race_year"] = df_idx["race_id"].astype(str).str[:4]

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_idx["field_speed_spread"].dropna(), kde=True, ax=ax[0])
    ax[0].set_title("field_speed_spread (hist)")
    sns.boxplot(data=df_idx, x="race_year", y="field_speed_spread", ax=ax[1])
    ax[1].set_title("field_speed_spread by race_year")
    plt.tight_layout()
    plt.show()

    # 仮説: 高 spread レースでは指数の「順位」が相対的に重要 — 散布図（馬行）
    rk = df_idx.groupby("race_id")[speed_col].rank(pct=True)
    df_idx["speed_pct_in_race"] = rk
    sample = df_idx.dropna(subset=["field_speed_spread", "speed_pct_in_race"]).sample(
        min(8000, len(df_idx)), random_state=0
    )
    plt.figure(figsize=(7, 5))
    sns.scatterplot(
        data=sample,
        x="field_speed_spread",
        y="speed_pct_in_race",
        alpha=0.25,
        hue="race_year",
    )
    plt.title("field_spread vs speed_pct_in_race (sample)")
    plt.tight_layout()
    plt.show()

## 4. （任意）race_performance 行データの年次分布

`data/features/race_performance/` に Parquet がある場合、主要指標のドリフトを眺める。

In [ ]:
rp_dir = REPO_ROOT / "data" / "features" / "race_performance"
rp_files = sorted(rp_dir.glob("*.parquet")) if rp_dir.exists() else []

if not rp_files:
    print("race_performance Parquet なし:", rp_dir)
else:
    frames = []
    for f in rp_files[:5]:
        frames.append(pd.read_parquet(f))
    rp = pd.concat(frames, ignore_index=True)
    print("columns:", list(rp.columns)[:25], "...")
    num = [c for c in rp.columns if rp[c].dtype.kind in "iufc"][:6]
    if num:
        if "race_id" in rp.columns:
            rp["_y"] = rp["race_id"].astype(str).str[:4]
        elif "date" in rp.columns:
            rp["_y"] = pd.to_datetime(rp["date"], errors="coerce").dt.year.astype("Int64")
        else:
            rp["_y"] = "all"
        m0 = num[0]
        plt.figure(figsize=(9, 4))
        if rp["_y"].nunique() > 1:
            sns.boxplot(data=rp, x="_y", y=m0)
            plt.title(f"race_performance: {m0} by year（サンプル）")
        else:
            sns.histplot(rp[m0].dropna(), kde=True)
            plt.title(f"race_performance: {m0}")
        plt.tight_layout()
        plt.show()
        if len(num) > 1:
            rp[num[:4]].hist(bins=30, figsize=(10, 6))
            plt.suptitle("race_performance numeric hists (first 4 cols)")
            plt.tight_layout()
            plt.show()

## 5. 次のメモ（発見の記録）

- 分布が年でズレる列 → adversarial / セグメント学習の候補。
- `field_*` が目的変数と弱いが安定 → **相互作用**（レース内 z-score 等）を次セルで試す。
- 採用候補は `feature_store.register_dataframe` で列登録し、`docs/modeling/feature_engineering_exploration.md` に 1 行追記。
- 学習試行は **MLflow** に `mlflow_run_id` を残す（`modeling_coding_strategy.md`）。

## 6. 10 サイクル一括探索（再現用）

`run_10_fe_cycles.py` が **仮説 → 特徴計算 → KS・Spearman → 短文メモ** を 10 回回し、  
`_run_output/cycles_report.md` と `cycles_metrics.json` に書き出す。  
`run_extended_fe_analysis.py` が **新規候補 6 本**を追加検証し `extended_analysis.md` に出力する。下のセルを実行。

In [ ]:
import subprocess
import sys
from pathlib import Path
from IPython.display import Markdown, display

for rel in (
    "notebooks/feature_engineering/run_10_fe_cycles.py",
    "notebooks/feature_engineering/run_extended_fe_analysis.py",
):
    script = Path(rel)
    r = subprocess.run([sys.executable, str(script)], cwd=REPO_ROOT, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        break
else:
    rep = REPO_ROOT / "notebooks/feature_engineering/_run_output/cycles_report.md"
    ext = REPO_ROOT / "notebooks/feature_engineering/_run_output/extended_analysis.md"
    if rep.exists():
        display(Markdown(rep.read_text(encoding="utf-8")[:8000]))
    if ext.exists():
        display(Markdown("\n---\n\n" + ext.read_text(encoding="utf-8")[:6000]))